<a href="https://colab.research.google.com/github/isratrimii/Neural-Networks/blob/main/Fake_Job_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install gradio

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from google.colab import files

Upload Dataset

In [4]:
uploaded = files.upload()

df = pd.read_csv("fake_job_postings.csv")

df = df[["description", "fraudulent"]]
df["description"] = df["description"].fillna("")

Saving fake_job_postings.csv to fake_job_postings.csv


**NLP Feature Engineering**

In [5]:
X_text = df["description"].astype(str).values
y = df["fraudulent"].values

tfidf = TfidfVectorizer(max_features=6000, stop_words="english")
X = tfidf.fit_transform(X_text).toarray()

**Train-Test Split**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

In [7]:
model = Sequential([
    Dense(512, activation="relu", input_shape=(6000,)),
    Dropout(0.5),

    Dense(256, activation="relu"),
    Dropout(0.4),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Train Model**

In [8]:
model.fit(
    X_train, y_train,
    epochs=12,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)

Epoch 1/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 31s 67ms/step - accuracy: 0.8868 - loss: 0.4891 - val_accuracy: 0.9123 - val_loss: 0.2544
Epoch 2/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - accuracy: 0.9381 - loss: 0.2019 - val_accuracy: 0.9259 - val_loss: 0.2063
Epoch 3/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.9693 - loss: 0.0897 - val_accuracy: 0.9588 - val_loss: 0.1447
Epoch 4/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 18s 51ms/step - accuracy: 0.9895 - loss: 0.0253 - val_accuracy: 0.9678 - val_loss: 0.2084
Epoch 5/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 17s 47ms/step - accuracy: 0.9896 - loss: 0.0380 - val_accuracy: 0.9560 - val_loss: 0.1956
Epoch 6/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 17s 49ms/step - accuracy: 0.9909 - loss: 0.0301 - val_accuracy: 0.9643 - val_loss: 0.2103
Epoch 7/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 16s 46ms/step - accuracy: 0.9927 - loss: 0.0182 - val_accuracy: 0.9685 - val_loss: 0.2133
Epoch 8/12
358/358 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.9958 - loss: 0.0152 - 

Evaluation

In [9]:
loss, acc = model.evaluate(X_test, y_test)
print("🔥 Accuracy:", acc)

y_pred = (model.predict(X_test) > 0.5).astype(int)

print("\n📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📄 Classification Report:")
print(classification_report(y_test, y_pred))

112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9698 - loss: 0.2695
🔥 Accuracy: 0.9697986841201782
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step

📊 Confusion Matrix:
[[3346   49]
 [  59  122]]

📄 Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      3395
           1       0.71      0.67      0.69       181

    accuracy                           0.97      3576
   macro avg       0.85      0.83      0.84      3576
weighted avg       0.97      0.97      0.97      3576



Smart Prediction Function

In [10]:
def predict_job(text):
    vec = tfidf.transform([text]).toarray()
    pred = model.predict(vec)[0][0]

    print("\n🧠 AI Score:", pred)

    if pred > 0.7:
        print("❌ Very Likely FAKE JOB")
    elif pred > 0.4:
        print("⚠️ Suspicious Job")
    else:
        print("✅ Likely REAL JOB")

**Test Example**

In [11]:
predict_job("Work from home job. No experience needed. Earn 5000 dollars easily.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

🧠 AI Score: 0.99994975
❌ Very Likely FAKE JOB


**Web App**

In [12]:
import gradio as gr

def predict_ui(text):
    vec = tfidf.transform([text]).toarray()
    pred = model.predict(vec)[0][0]

    if pred > 0.7:
        return "❌ FAKE JOB"
    elif pred > 0.4:
        return "⚠️ SUSPICIOUS JOB"
    else:
        return "✅ REAL JOB"

app = gr.Interface(
    fn=predict_ui,
    inputs="text",
    outputs="text",
    title="Fake Job Detection AI",
    description="Paste job description and check if it's real or fake"
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76925ac79a3d78b165.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
